# Previsão de Doenças Cardíacas com Azure Machine Learning

**Dataset:** Heart Disease UCI  
**Objetivo:** Classificar pacientes como saudáveis (0) ou com doença cardíaca (1)  
**Algoritmos:** Random Forest e XGBoost  
**Métricas:** AUC-ROC e F1-Score

## 1. Importações e Configuração

In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split, GridSearchCV, cross_val_score, StratifiedKFold
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    f1_score, roc_auc_score, accuracy_score,
    classification_report, confusion_matrix, RocCurveDisplay
)
from xgboost import XGBClassifier
import mlflow
import mlflow.sklearn
import mlflow.xgboost

RANDOM_STATE = 42
print("Bibliotecas carregadas com sucesso.")

## 2. Carregamento dos Dados (Azure Blob Storage)

In [ ]:
try:
    # Caminho do Azure ML Datastore (Blob Storage)
    path = "azureml://datastores/workspaceblobstore/paths/heart.csv"
    df = pd.read_csv(path)
    print("Carregado via Azure Blob Storage")
except Exception:
    df = pd.read_csv("../data/heart.csv")
    print("Carregado localmente (fallback)")

print(f"Shape: {df.shape}")
df.head()

## 3. EDA — Análise Exploratória dos Dados

### 3.1 Estatísticas Descritivas e Qualidade dos Dados

In [ ]:
print("=== Tipos e shape ===")
print(df.dtypes)
print(f"\nShape: {df.shape}")

print("\n=== Valores nulos ===")
nulls = df.isnull().sum()
print(nulls[nulls > 0] if nulls.sum() > 0 else "Nenhum valor nulo encontrado.")

print("\n=== Estatísticas descritivas ===")
df.describe().T

### 3.2 Distribuição da Variável-Alvo

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

counts = df['target'].value_counts()
axes[0].bar(['Saudável (0)', 'Doente (1)'], counts.values, color=['#2ecc71', '#e74c3c'])
axes[0].set_title('Distribuição da Variável-Alvo')
axes[0].set_ylabel('Quantidade')
for i, v in enumerate(counts.values):
    axes[0].text(i, v + 1, str(v), ha='center', fontweight='bold')

axes[1].pie(counts.values, labels=['Saudável', 'Doente'],
            colors=['#2ecc71', '#e74c3c'], autopct='%1.1f%%', startangle=90)
axes[1].set_title('Proporção das Classes')

plt.tight_layout()
plt.show()

### 3.3 Distribuição das Features por Classe

In [ ]:
numeric_cols = ['age', 'trestbps', 'chol', 'thalach', 'oldpeak']

fig, axes = plt.subplots(2, 3, figsize=(15, 8))
axes = axes.flatten()

for i, col in enumerate(numeric_cols):
    for target_val, color, label in [(0, '#2ecc71', 'Saudável'), (1, '#e74c3c', 'Doente')]:
        axes[i].hist(df[df['target'] == target_val][col], bins=20,
                     alpha=0.6, color=color, label=label)
    axes[i].set_title(f'Distribuição: {col}')
    axes[i].legend()

axes[-1].set_visible(False)
plt.suptitle('Distribuição das Features Numéricas por Classe', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

### 3.4 Boxplots — Detecção de Outliers

In [ ]:
fig, axes = plt.subplots(1, len(numeric_cols), figsize=(16, 5))

for i, col in enumerate(numeric_cols):
    df.boxplot(column=col, by='target', ax=axes[i])
    axes[i].set_title(col)
    axes[i].set_xlabel('Target')

plt.suptitle('Boxplots por Classe (0=Saudável, 1=Doente)', fontsize=13)
plt.tight_layout()
plt.show()

# Contagem de outliers via IQR
print("=== Outliers por feature (método IQR) ===")
for col in numeric_cols:
    Q1, Q3 = df[col].quantile(0.25), df[col].quantile(0.75)
    IQR = Q3 - Q1
    outliers = ((df[col] < Q1 - 1.5 * IQR) | (df[col] > Q3 + 1.5 * IQR)).sum()
    print(f"  {col}: {outliers} outlier(s)")

### 3.5 Mapa de Correlação

In [ ]:
plt.figure(figsize=(12, 9))
corr = df.corr()
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=True, fmt='.2f', cmap='coolwarm',
            center=0, square=True, linewidths=0.5)
plt.title('Mapa de Correlação entre Features', fontsize=14)
plt.tight_layout()
plt.show()

print("\nCorrelações mais altas com 'target':")
print(corr['target'].abs().sort_values(ascending=False).drop('target'))

## 4. Pré-processamento e Feature Engineering

In [ ]:
df_fe = df.copy()

# Feature Engineering
# Razão colesterol / idade (risco relativo por faixa etária)
df_fe['chol_per_age'] = df_fe['chol'] / df_fe['age']

# Faixa etária (binning)
df_fe['age_group'] = pd.cut(df_fe['age'], bins=[0, 40, 55, 70, 100],
                             labels=[0, 1, 2, 3]).astype(int)

# Pressão arterial elevada (flag binária: > 140 mmHg)
df_fe['high_bp'] = (df_fe['trestbps'] > 140).astype(int)

# Frequência cardíaca relativa à idade máxima esperada (220 - idade)
df_fe['hr_reserve'] = df_fe['thalach'] / (220 - df_fe['age'])

print("Features criadas:", ['chol_per_age', 'age_group', 'high_bp', 'hr_reserve'])
print(f"Shape após feature engineering: {df_fe.shape}")
df_fe[['age', 'chol', 'thalach', 'chol_per_age', 'age_group', 'high_bp', 'hr_reserve']].head()

In [ ]:
X = df_fe.drop('target', axis=1)
y = df_fe['target']

# Normalização
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)
X_scaled = pd.DataFrame(X_scaled, columns=X.columns)

# Split estratificado (mantém proporção das classes)
X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y
)

print(f"Treino: {X_train.shape[0]} amostras | Teste: {X_test.shape[0]} amostras")
print(f"Distribuição treino — 0: {(y_train==0).sum()} | 1: {(y_train==1).sum()}")
print(f"Distribuição teste  — 0: {(y_test==0).sum()} | 1: {(y_test==1).sum()}")

## 5. Treinamento com Otimização de Hiperparâmetros

### 5.1 Random Forest — GridSearchCV

In [ ]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

rf_params = {
    'n_estimators': [50, 100, 200],
    'max_depth': [3, 5, None],
    'min_samples_split': [2, 5]
}

rf_grid = GridSearchCV(
    RandomForestClassifier(random_state=RANDOM_STATE),
    rf_params, cv=cv, scoring='f1', n_jobs=-1, verbose=1
)
rf_grid.fit(X_train, y_train)

print(f"Melhores hiperparâmetros RF: {rf_grid.best_params_}")
print(f"Melhor F1 (CV treino): {rf_grid.best_score_:.4f}")

### 5.2 XGBoost — GridSearchCV

In [ ]:
xgb_params = {
    'n_estimators': [100, 200],
    'max_depth': [3, 5],
    'learning_rate': [0.05, 0.1],
    'subsample': [0.8, 1.0]
}

xgb_grid = GridSearchCV(
    XGBClassifier(eval_metric='logloss', random_state=RANDOM_STATE),
    xgb_params, cv=cv, scoring='f1', n_jobs=-1, verbose=1
)
xgb_grid.fit(X_train, y_train)

print(f"Melhores hiperparâmetros XGB: {xgb_grid.best_params_}")
print(f"Melhor F1 (CV treino): {xgb_grid.best_score_:.4f}")

## 6. Avaliação e Validação

### 6.1 Cross-Validation — AUC e F1 nos dois modelos

In [ ]:
rf_best = rf_grid.best_estimator_
xgb_best = xgb_grid.best_estimator_

results_cv = {}
for name, model in [('Random Forest', rf_best), ('XGBoost', xgb_best)]:
    f1_scores  = cross_val_score(model, X_train, y_train, cv=cv, scoring='f1')
    auc_scores = cross_val_score(model, X_train, y_train, cv=cv, scoring='roc_auc')
    results_cv[name] = {
        'F1 médio (CV)':  f1_scores.mean(),
        'F1 std':         f1_scores.std(),
        'AUC médio (CV)': auc_scores.mean(),
        'AUC std':        auc_scores.std()
    }
    print(f"{name}: F1={f1_scores.mean():.4f} ± {f1_scores.std():.4f} | "
          f"AUC={auc_scores.mean():.4f} ± {auc_scores.std():.4f}")

pd.DataFrame(results_cv).T

### 6.2 Avaliação no Conjunto de Teste

In [ ]:
results_test = {}

for name, model in [('Random Forest', rf_best), ('XGBoost', xgb_best)]:
    y_pred = model.predict(X_test)
    y_proba = model.predict_proba(X_test)[:, 1]

    results_test[name] = {
        'Accuracy':  accuracy_score(y_test, y_pred),
        'F1-Score':  f1_score(y_test, y_pred),
        'AUC-ROC':   roc_auc_score(y_test, y_proba)
    }

print("=== Comparação no Conjunto de Teste ===")
df_results = pd.DataFrame(results_test).T.round(4)
print(df_results)
df_results

### 6.3 Relatório de Classificação e Matrizes de Confusão

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

for i, (name, model) in enumerate([('Random Forest', rf_best), ('XGBoost', xgb_best)]):
    y_pred = model.predict(X_test)
    print(f"\n=== {name} ===")
    print(classification_report(y_test, y_pred, target_names=['Saudável', 'Doente']))

    cm = confusion_matrix(y_test, y_pred)
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[i],
                xticklabels=['Saudável', 'Doente'],
                yticklabels=['Saudável', 'Doente'])
    axes[i].set_title(f'Matriz de Confusão — {name}')
    axes[i].set_ylabel('Real')
    axes[i].set_xlabel('Predito')

plt.tight_layout()
plt.show()

### 6.4 Curvas ROC

In [ ]:
fig, ax = plt.subplots(figsize=(8, 6))

for name, model in [('Random Forest', rf_best), ('XGBoost', xgb_best)]:
    RocCurveDisplay.from_estimator(model, X_test, y_test, ax=ax, name=name)

ax.plot([0, 1], [0, 1], 'k--', label='Aleatório (AUC = 0.50)')
ax.set_title('Curvas ROC — Comparação dos Modelos')
ax.legend()
plt.tight_layout()
plt.show()

### 6.5 Importância das Features

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

for i, (name, model) in enumerate([('Random Forest', rf_best), ('XGBoost', xgb_best)]):
    importances = pd.Series(model.feature_importances_, index=X.columns)
    importances.sort_values().plot(kind='barh', ax=axes[i], color='steelblue')
    axes[i].set_title(f'Importância das Features — {name}')
    axes[i].set_xlabel('Importância')

plt.tight_layout()
plt.show()

## 7. Registro dos Modelos no Azure ML (MLflow)

In [ ]:
import os, tempfile

mlflow.set_experiment("heart-disease-experiment")

# Determina o melhor modelo pelo AUC no teste
best_model_name = max(results_test, key=lambda k: results_test[k]['AUC-ROC'])
print(f"Melhor modelo pelo AUC-ROC no teste: {best_model_name}")

models_to_log = [
    ('random_forest', rf_best, 'Random Forest', rf_grid.best_params_),
    ('xgboost', xgb_best, 'XGBoost', xgb_grid.best_params_)
]

for artifact_name, model, display_name, best_params in models_to_log:
    with mlflow.start_run(run_name=display_name):
        # Hiperparâmetros
        mlflow.log_params(best_params)

        # Métricas no conjunto de teste
        mlflow.log_metric('accuracy',  results_test[display_name]['Accuracy'])
        mlflow.log_metric('f1_score',  results_test[display_name]['F1-Score'])
        mlflow.log_metric('roc_auc',   results_test[display_name]['AUC-ROC'])

        # Métricas de CV
        mlflow.log_metric('cv_f1_mean',  results_cv[display_name]['F1 médio (CV)'])
        mlflow.log_metric('cv_auc_mean', results_cv[display_name]['AUC médio (CV)'])

        # Tag para identificar o melhor modelo
        mlflow.set_tag('best_model', str(display_name == best_model_name))

        # Salva localmente e envia como artefato (compatível com Azure ML MLflow server)
        with tempfile.TemporaryDirectory() as tmp_dir:
            model_dir = os.path.join(tmp_dir, artifact_name)
            if artifact_name == 'xgboost':
                mlflow.xgboost.save_model(model, model_dir)
            else:
                mlflow.sklearn.save_model(model, model_dir)
            mlflow.log_artifacts(model_dir, artifact_path=artifact_name)

        print(f"[{display_name}] registrado — AUC: {results_test[display_name]['AUC-ROC']:.4f} "
              f"| F1: {results_test[display_name]['F1-Score']:.4f}")


## 8. Conclusão

In [ ]:
print("=" * 55)
print("           RESUMO FINAL DO EXPERIMENTO")
print("=" * 55)
print(f"{'Modelo':<20} {'Accuracy':>10} {'F1-Score':>10} {'AUC-ROC':>10}")
print("-" * 55)
for model_name, metrics in results_test.items():
    marker = " <-- melhor" if model_name == best_model_name else ""
    print(f"{model_name:<20} {metrics['Accuracy']:>10.4f} "
          f"{metrics['F1-Score']:>10.4f} {metrics['AUC-ROC']:>10.4f}{marker}")
print("=" * 55)
print(f"\nModelo vencedor: {best_model_name}")
print("Ambos os modelos foram registrados no Azure ML via MLflow.")